# 03 · Lakeflow Declarative Pipeline ⭐

En este lab usaremos **Genie Code** para generar el código fuente de una **Lakeflow Spark Declarative Pipeline** que construye el medallion completo del caso CFE:

```
BRONZE   54 streaming tables     ← Auto Loader lee cada Parquet del Volume (crudo)
SILVER   tablas clave limpias    ← Expectations (WARN/DROP/FAIL) + tipado + dedup
GOLD     modelo estrella         ← Tabla de hechos + 5 dimensiones para análisis
```

**Alcance:** ingesta de las **54 tablas** fuente, limpieza con expectations en las tablas principales,
y un **modelo estrella** con `gold_fact_obras` como hecho central y dimensiones de proyecto,
estado, tipo de obra, residencia regional y estado de proyecto.

> ⚠️ El código de una Declarative Pipeline **no se corre celda por celda** — se adjunta a una *pipeline* que Databricks orquesta.

In [0]:
%run ./_setup

## 1. Arquitectura: Medallion → Modelo Estrella

### Bronze (54 streaming tables)
Cada archivo Parquet en `DATA_PATH` se ingiere como streaming table con Auto Loader.
Ejemplo: `tbobras.parquet` → `bronze_tbobras`, `ctestadosrepublica.parquet` → `bronze_ctestadosrepublica`, etc.

### Silver (tablas clave con calidad)
| Tabla | Fuente | Transformaciones |
|---|---|---|
| `silver_obras` | `bronze_tbobras` | Dedup `IdObra`, castear fechas a DATE, expectations |
| `silver_proyectos` | `bronze_tbproyectos` | Dedup `IdProyecto`, castear fechas, expectations |
| `silver_obralinea` | `bronze_obralinea` | Expectations en Kilometros |
| `silver_obrasubestacion` | `bronze_obrasubestacion` | Expectations en MVA |

### Gold (modelo estrella)
| Tabla | Tipo | Descripción |
|---|---|---|
| **`gold_fact_obras`** | Hecho | Grain: 1 fila/obra. Medidas: ImporteContratoMxn, ICM, CPTT, AvanceActualConstruccion, Kilometros, MVA |
| `gold_dim_proyectos` | Dimensión | Nombre, Alcance, Cliente, Contratista, MontoPMC, NumeroContrato |
| `gold_dim_estados` | Dimensión | NombreEstado, Abreviatura (32 estados) |
| `gold_dim_tipos_obra` | Dimensión | TipoObra: Línea / Subestación |
| `gold_dim_residencias` | Dimensión | 7 residencias regionales (Nombre, Siglas) |
| `gold_dim_estados_proyecto` | Dimensión | 6 estados de proyecto |

## 2. Expectations por tabla

| Modo | Decorador | Efecto |
|---|---|---|
| **WARN** | `@dlt.expect` | Registra violación, **no descarta** |
| **DROP** | `@dlt.expect_or_drop` | **Descarta** filas inválidas |
| **FAIL** | `@dlt.expect_or_fail` | **Detiene** la pipeline |

### silver_obras
* **FAIL:** `IdObra IS NOT NULL AND IdProyecto IS NOT NULL`
* **DROP:** `NombreObra IS NOT NULL`
* **WARN:** `ImporteContratoMxn >= 0`, `AvanceActualConstruccion BETWEEN 0 AND 100`

### silver_proyectos
* **FAIL:** `IdProyecto IS NOT NULL`
* **DROP:** `Nombre IS NOT NULL`
* **WARN:** `IdEstadoRep BETWEEN 1 AND 32`, `MontoPMC >= 0`

### silver_obralinea
* **FAIL:** `IdObra IS NOT NULL`
* **WARN:** `Kilometros >= 0`, `Circuitos > 0`

### silver_obrasubestacion
* **FAIL:** `IdObra IS NOT NULL`
* **WARN:** `MVA >= 0`, `Alimentadores > 0`

## 3. Actividad: genera tu pipeline con Genie Code

### Pasos:
1. **Abre un notebook nuevo** (File → New → Notebook) con lenguaje Python.
2. **Ejecuta la celda de abajo** para obtener el prompt con tus variables.
3. **Copia el prompt** impreso en la salida.
4. **Abre Genie Code** (`Cmd+I` / `Ctrl+I`) en el notebook nuevo.
5. **Pega el prompt** y genera el código.
6. Revisa que incluya:
   * 54 streaming tables en bronce (patrón DRY, sin repetir código)
   * Expectations (WARN/DROP/FAIL) en las 4 tablas silver
   * Tabla de hechos `gold_fact_obras` con JOINs
   * 5 tablas de dimensión
7. **No lo ejecutes** — es código de pipeline.

### Crear y ejecutar la pipeline:
1. Menú **Jobs & Pipelines → Create → ETL Pipeline**.
2. **Source code:** tu notebook recién generado.
3. **Destino:** catálogo `dipidev` y tu esquema personal.
4. **Serverless** ✔ · Pipeline mode: **Triggered**.
5. **Start** — observa el DAG de dependencias y los resultados de expectations.

In [0]:
user_alias = spark.sql("SELECT current_user()").first()[0].split("@")[0].replace(".", "_")
pipeline_name = f"pipeline_cfe_{user_alias}"

# Lista de los 54 archivos Parquet disponibles
parquet_files = [f.name.replace(".parquet", "") for f in dbutils.fs.ls(DATA_PATH) if f.name.endswith(".parquet")]
parquet_list = ", ".join(parquet_files)

prompt = f"""
Genera una Lakeflow Declarative Pipeline en SQL.
Comentario al inicio: # Pipeline: {pipeline_name} | Autor: {user_alias}

===== CONFIGURACIÓN =====
Catálogo: {CATALOG}
Esquema: {WORK_SCHEMA}
Carpeta de Parquets: {DATA_PATH}
Ruta configurable: spark.conf.get("cfe.source_path", "{DATA_PATH}")

===== BRONZE: 54 MATERIALIZED VIEWS =====
Crea una materialized view por cada archivo Parquet en la carpeta.
Los archivos son: {parquet_list}

Para cada uno:
- Nombre: bronze_<nombre_archivo> (ej: tbobras.parquet → bronze_tbobras)
- Usa spark.read.parquet() (NO cloudFiles/Auto Loader, porque cada .parquet es un archivo individual, no un directorio)
- Agrega columna _ingestion_timestamp con current_timestamp()
- table_properties: {{"quality": "bronze"}}
- comment descriptivo: "Bronze: ingesta cruda de <nombre>"

NOTA: los Parquets originales tienen una columna `_rescued` (no `_rescued_data`). Excluir `_rescued` en silver/gold.

IMPORTANTE: para no repetir código, usa un patrón DRY con un loop o función factory en Python
que genere las 54 tablas dinámicamente con @dlt.table().
Las tablas silver y gold deben ir en un archivo SQL aparte (transformations/silver_gold.sql)
porque un notebook Python no puede tener celdas SQL en una pipeline.

===== SILVER: TABLAS CLAVE CON CALIDAD =====

### silver_obras (desde bronze_tbobras, 318 filas, 50 cols)
- Dedup por IdObra (quedarse con la fila más reciente por _ingestion_timestamp)
- Castear a DATE: FechaInicioProgramada, FechaTerminacionProgramada, FechaInicioReal, FechaTerminacionReal
- Liquid Clustering por (IdProyecto, IdTipoObra)
- Expectations:
  * FAIL: "pk_not_null" → IdObra IS NOT NULL AND IdProyecto IS NOT NULL
  * DROP: "nombre_presente" → NombreObra IS NOT NULL
  * WARN: "importe_valido" → ImporteContratoMxn >= 0
  * WARN: "avance_rango" → AvanceActualConstruccion BETWEEN 0 AND 100

### silver_proyectos (desde bronze_tbproyectos, 78 filas, 72 cols)
- Dedup por IdProyecto
- Castear a DATE: FechaInicioProg, FechaTerminacionProg, FechaInicioReal, FechaTerminacionReal
- SIN Liquid Clustering (IdEstadoRep e IdTipoProyecto están más allá de las primeras 32 columnas y no tienen stats)
- Expectations:
  * FAIL: "pk_not_null" → IdProyecto IS NOT NULL
  * DROP: "nombre_presente" → Nombre IS NOT NULL
  * WARN: "estado_valido" → IdEstadoRep BETWEEN 1 AND 32
  * WARN: "monto_valido" → MontoPMC >= 0

### silver_obralinea (desde bronze_obralinea, 118 filas)
- Expectations:
  * FAIL: "pk_not_null" → IdObra IS NOT NULL
  * WARN: "km_valido" → Kilometros >= 0
  * WARN: "circuitos_valido" → Circuitos > 0

### silver_obrasubestacion (desde bronze_obrasubestacion, 200 filas)
- Expectations:
  * FAIL: "pk_not_null" → IdObra IS NOT NULL
  * WARN: "mva_valido" → MVA >= 0
  * WARN: "alimentadores_valido" → Alimentadores > 0

===== GOLD: MODELO ESTRELLA =====

### gold_fact_obras (tabla de hechos, grain: 1 fila por obra)
Join de silver_obras con:
- silver_proyectos (por IdProyecto) para obtener IdEstadoRep, IdResidenciaRegional, IdEstadoProyecto
- silver_obralinea (LEFT JOIN por IdObra) para Kilometros, Circuitos
- silver_obrasubestacion (LEFT JOIN por IdObra) para MVA, MVAr, Alimentadores

Columnas:
- IdObra (PK), NombreObra, SituacionActual, BreveDescripcion
- IdProyecto (FK), IdTipoObra (FK), IdEstadoRep (FK), IdResidenciaRegional (FK), IdEstadoProyecto (FK)
- ImporteContratoMxn, ICM, CPTT, CN, ACB (medidas financieras)
- AvanceActualConstruccion, AvanceGeneralAP, ActividadesPrevias (medidas de avance)
- FechaInicioProgramada, FechaTerminacionProgramada, FechaInicioReal (fechas DATE)
- kV (voltaje)
- Kilometros, KmC, Circuitos (de obralinea, NULL si es subestación)
- MVA, MVAr, Alimentadores (de obrasubestacion, NULL si es línea)
- Liquid Clustering por (IdProyecto, IdEstadoRep)
- table_properties: {{"quality": "gold"}}

### gold_dim_proyectos (desde silver_proyectos)
- IdProyecto (PK), Nombre, Alcance, Cliente, Contratista, NumeroContrato,
  MontoPMC, FechaInicioProg, FechaTerminacionProg, Latitud, Longitud

### gold_dim_estados (desde bronze_ctestadosrepublica)
- IdEstadoRep (PK), NombreEstado, Abreviatura

### gold_dim_tipos_obra (desde bronze_cttipoobra)
- IdTipoObra (PK), TipoObra

### gold_dim_residencias (desde bronze_ctresidenciasregionales)
- IdResidenciaRegional (PK), Nombre, Siglas

### gold_dim_estados_proyecto (desde bronze_ctestadosproyectos)
- IdEstadoProyecto (PK), Nombre

Todas las tablas gold con comment descriptivo y table_properties quality = gold.
No incluir la columna _rescued en las tablas gold.

===== NOTAS TÉCNICAS =====
1. Bronze en Python (notebook), Silver+Gold en SQL (archivo .sql separado)
   → DLT ignora celdas %sql en notebooks Python («magic commands not supported»)
2. spark.read.parquet() en vez de cloudFiles porque cada .parquet es un archivo, no un directorio
3. La columna de rescued data se llama `_rescued` (viene del Parquet original), no `_rescued_data`
4. Liquid Clustering solo funciona con columnas que tengan stats (primeras 32 del esquema)
"""

print(prompt)